# Chapter 12 — Prompt Templates

**Companion to *Pi Agents*** · [`Pi Agents` chapter 12](https://programmer.ie/books/pi/12-chapter/)

| | |
|---|---|
| Chapter | `12-chapter.md` · weight 12 · `/books/pi/12-chapter/` |
| Notebook | `notebooks/pi/12-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Send `/review` through a real session and read **the message the provider
actually received**. What was substituted, what was not, and who saw the raw text
before expansion?

A template is text plus a substitution table. Everything interesting is in the
edges: defaults, quoting, and what happens when an extension registers a command of
the same name.


## What this notebook establishes

- The **default** `${1:-correctness, security, and error handling}` applies with no argument; an argument replaces it entirely.
- Shell-like quoting is honoured: `/review "API compatibility"` produces one argument, not two.
- The substitution table is `$1`, `$2`, `$@`, `$ARGUMENTS`, `${@:N}` and `${@:N:L}` — all six, in one request, asserted exactly.
- An `input` handler sees the **raw** text `/review "API compatibility"`, before expansion; an extension command of the same name preempts the template entirely, and the model is never asked (0 requests).

## What this notebook does **not** establish

- **A template can ask the model to run a command but cannot ensure it ran.** `metadata/12-chapter.yaml` records that limit; nothing here changes it.
- **Prompt templates are discovered non-recursively** — direct `.md` children of the conventional directories. Skills are recursive; that contrast is chapter 13.
- **Command/extension collision resolution is left open** by the chapter (`ch12-q2`). What is shown above is the observed behaviour: the extension wins and the model is not called.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — a real template in a real agent directory, sent through a real session, with the provider's request captured.
* **Evidence scope:** `in_process_runtime`. Expansion, discovery and precedence are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(12))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/12-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: what the model receives

The driver loads a template with an argument hint and a default, sends `/review`
twice — once bare, once with a quoted argument — and captures two different things
per run:

* `rawSeen` — what an `input` handler was given, which is the text as typed;
* `sentUser` — the `user` message content in the request the provider received.

If those two were the same string, the substitution would not be happening.

In [3]:
TEMPLATES = pinb.driver_source("ch12-templates.ts")
templates = pinb.driver(TEMPLATES, label="ch12-templates", timeout=300)
print("template body:")
print("    Review the uncommitted git changes.")
print("    Focus on ${1:-correctness, security, and error handling}.")
print()
rows = [
    ["/review", templates["default"]["rawSeen"], templates["default"]["sentUser"]],
    ['/review "API compatibility"', templates["argument"]["rawSeen"], templates["argument"]["sentUser"]],
]
print(pinb.md_table(
    ["typed", "raw text an input handler saw", "the user message the provider received"],
    [[typed, raw, sent.replace(chr(10), " / ")] for typed, raw, sent in rows],
))

template body:
    Review the uncommitted git changes.
    Focus on ${1:-correctness, security, and error handling}.

| typed | raw text an input handler saw | the user message the provider received |
|---|---|---|
| /review | /review | Review the uncommitted git changes. / Focus on correctness, security, and error handling. |
| /review "API compatibility" | /review "API compatibility" | Review the uncommitted git changes. / Focus on API compatibility. |


In [4]:
d, a = templates["default"], templates["argument"]

pinb.show_checks([
    pinb.check("the default applies with no argument",
               "correctness, security, and error handling" in d["sentUser"], "default present"),
    pinb.check("an argument replaces the default entirely",
               "API compatibility" in a["sentUser"] and "correctness" not in a["sentUser"],
               a["sentUser"].splitlines()[-1]),
    pinb.check("a quoted phrase is one argument",
               a["sentUser"].rstrip().endswith("Focus on API compatibility."), "no split"),
    pinb.check("the input handler sees the raw text, before expansion",
               a["rawSeen"] == '/review "API compatibility"', repr(a["rawSeen"])),
    pinb.check("the raw text still contains the quotes the model never sees",
               '"' in a["rawSeen"] and '"' not in a["sentUser"], "quotes consumed by expansion"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------------  -----------------------------
PASS  the default applies with no argument  default present
PASS  an argument replaces the default entirely  Focus on API compatibility.
PASS  a quoted phrase is one argument  no split
PASS  the input handler sees the raw text, before expansion  '/review "API compatibility"'
PASS  the raw text still contains the quotes the model never sees  quotes consumed by expansion

5/5 assertions held.


## The substitution table, all six at once

One template, six forms, one request. The expected string is fixed, so this is a
real assertion rather than a demonstration: if Pi's parser changed a form, this row
changes.

In [5]:
table = templates["substitutionTable"]
print("template body:")
print("    A=[$1] B=[$2] ALL=[$@] ARGUMENTS=[$ARGUMENTS] FROM2=[${@:2}] 2of2=[${@:2:2}] NONE=[$3]")
print("invoked as: /t a b c")
print()
print("the provider received:")
print("   ", table)
print()
expected = "A=[a] B=[b] ALL=[a b c] ARGUMENTS=[a b c] FROM2=[b c] 2of2=[b c] NONE=[c]"

pinb.show_checks([
    pinb.check("every form in the table substituted as documented",
               table == expected, "exact match" if table == expected else repr(table)),
    pinb.check("$1 and $2 are positional", "A=[a] B=[b]" in table, "positional"),
    pinb.check("$@ and $ARGUMENTS are the whole argument list",
               "ALL=[a b c]" in table and "ARGUMENTS=[a b c]" in table, "both present"),
    pinb.check("${@:N} drops the first N, and ${@:N:L} also takes a length",
               "FROM2=[b c]" in table and "2of2=[b c]" in table, "slice forms"),
    pinb.check("a positional beyond the arguments falls through to the next argument",
               "NONE=[c]" in table, "$3 resolved to c rather than being left literal"),
])

template body:
    A=[$1] B=[$2] ALL=[$@] ARGUMENTS=[$ARGUMENTS] FROM2=[${@:2}] 2of2=[${@:2:2}] NONE=[$3]
invoked as: /t a b c

the provider received:
    A=[a] B=[b] ALL=[a b c] ARGUMENTS=[a b c] FROM2=[b c] 2of2=[b c] NONE=[c]

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------  -----------------------------------------------
PASS  every form in the table substituted as documented  exact match
PASS  $1 and $2 are positional  positional
PASS  $@ and $ARGUMENTS are the whole argument list  both present
PASS  ${@:N} drops the first N, and ${@:N:L} also takes a length  slice forms
PASS  a positional beyond the arguments falls through to the next argument  $3 resolved to c rather than being left literal

5/5 assertions held.


(The seventh form, `${1:-default}`, is the one asserted in the first table — the
default applied when the argument was absent and was replaced when one was given.)

## Contrasting case: an extension command of the same name

Preemption is total. The handler runs, the template does not, and — the number that
matters — **the model is never asked**: `requests == 0`.

In [6]:
withCommand = templates["withCommand"]
print(pinb.md_table(
    ["/review with an extension command of the same name registered", "the provider received"],
    [["user message sent to the model", withCommand["sentUser"] or "(nothing)"],
     ["provider requests", withCommand["requests"]]],
))

pinb.show_checks([
    pinb.check("the template was discovered", "review" in withCommand["templates"], ", ".join(withCommand["templates"])),
    pinb.check("but the extension command preempts it", withCommand["sentUser"] == "", "no user message"),
    pinb.check("and the model was never asked", withCommand["requests"] == 0, str(withCommand["requests"])),
])

| /review with an extension command of the same name registered | the provider received |
|---|---|
| user message sent to the model | (nothing) |
| provider requests | 0 |
== Assertions ==
  assertion  observed
----  -------------------------------------  ---------------
PASS  the template was discovered  review
PASS  but the extension command preempts it  no user message
PASS  and the model was never asked  0

3/3 assertions held.


## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch12-templates/templates.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch12-templates/templates.test.ts', show="template")

$ node --test ch12-templates/templates.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 997 ms

  tests matching 'template':
    ok   an extension command with the same name wins over the template


## Your turn: predict, then run

**Predict first.** Write a template with `${1:-A} and ${2:-B}` and invoke it with
`/x first`. What reaches the model — and how many words is it?

**Then change one input.** `TEMPLATE_BODY` below is what the driver writes. Add
`${@}` next to `$1` and re-run the table.

**Predict the boundary.** Why is `${3}` in the table above left as the literal `$3`
in one reading and as `c` in another? Decide which behaviour you would want for a
template whose third argument is genuinely optional, and say what the current
behaviour costs you.

In [8]:
TABLE_ROW = "A=[$1] B=[$2] ALL=[$@] ARGUMENTS=[$ARGUMENTS] FROM2=[${@:2}] 2of2=[${@:2:2}] NONE=[$3]"
import re


def substitute(body: str, args: list[str]) -> str:
    """A Python restatement of the documented table, for inspection only.

    The authoritative result is the request Pi built, shown two cells above.
    """
    out = body
    for i, a in enumerate(args, 1):
        out = out.replace(f"${i}", a)
        out = re.sub(r"\$\{%d(?::-([^}]*))?\}" % i, a or (r"\1" or ""), out)
    out = out.replace("$@", " ".join(args)).replace("$ARGUMENTS", " ".join(args))
    out = re.sub(r"\$\{@:(\d+)(?::(\d+))?\}",
                 lambda m: " ".join(args[int(m.group(1)) - 1:(int(m.group(1)) - 1 + int(m.group(2))) if m.group(2) else None]), out)
    return out


print("template:", TABLE_ROW)
print("args    : a b c")
print("result  :", substitute(TABLE_ROW, ["a", "b", "c"]))
print("observed:", templates["substitutionTable"])

assert substitute(TABLE_ROW, ["a", "b", "c"]) == templates["substitutionTable"], (
    "the restated table must agree with what Pi actually built")
print()
print("held: the rule, restated in Python, produces byte-identical output to Pi")
print()
print("Now change one input. What does ${3} cost you when the third argument is optional?")

template: A=[$1] B=[$2] ALL=[$@] ARGUMENTS=[$ARGUMENTS] FROM2=[${@:2}] 2of2=[${@:2:2}] NONE=[$3]
args    : a b c
result  : A=[a] B=[b] ALL=[a b c] ARGUMENTS=[a b c] FROM2=[b c] 2of2=[b c] NONE=[c]
observed: A=[a] B=[b] ALL=[a b c] ARGUMENTS=[a b c] FROM2=[b c] 2of2=[b c] NONE=[c]

held: the rule, restated in Python, produces byte-identical output to Pi

Now change one input. What does ${3} cost you when the third argument is optional?


## Interpretation

A prompt template is three things stacked, and each one can fail quietly:

1. **Discovery** — conventional directories, direct `.md` children only. A nested
   template is not found, with no error.
2. **Precedence** — an extension command of the same name wins, and the model is
   never asked. Cheap to check (`requests == 0`) and invisible if you only read the
   output.
3. **Expansion** — the table above, applied to the raw text.

The practical rules that follow:

* **Put the instruction in the template, not in the habit.** A template is a file a
  team can review; a habit is a sentence in somebody's head.
* **Do not use a template to assert something happened.** It can *ask* the model to
  run a check. It cannot know whether it did. That is the limit
  `metadata/12-chapter.yaml` records, and it is chapter 16's subject.
* **Check nesting before syntax.** A template one directory too deep is not a
  template with a bug; it is not a template.

## Sources, execution mode and limitations

**Execution mode:** **Run** — four real sessions with a real template in a real agent directory, capturing the provider's request each time.

**Evidence scope:** `in_process_runtime`. Expansion, discovery and command precedence are Pi's.

### What was read or run

- **Ran** `drivers/ch12-templates.ts`: four sessions, each with its own temporary agent directory and its own `prompts/*.md`.
- **Captured the `user` message** out of the provider request in a faux response factory, so the assertion is about what the provider received.
- **Ran** `ch12-templates/templates.test.ts`.
- **Read** `examples/evidence.json`, chapter 12 rows (7 claims).

### Limitations

- **Substitution is not sandboxing.** A template body is text handed to a model. Nothing here establishes that the model will do what the template says, only that it was told it in this exact form.
- **The Python `substitute` in the exercise is a restatement**, not an implementation, and its only claim is that it agrees byte-for-byte with what Pi built in the cell above. The authority is the observed request.
- **Command precedence is observed, not specified.** The chapter leaves `ch12-q2` open; what is shown is what this release does.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
